# 预测试题目筛选与难度分级

本 Notebook 完成四项工作：

1. 用严格完全识别率识别天花板题和地板题；
2. 检查完整题库选项中的零选择选项；
3. 识别相对同素材模态而言作答过久的题目；
4. 在剔除极端题后计算题目难度并分级。

统计结果是筛选依据，删除题目或选项前仍需内容复核。原始 CSV 不会被修改。

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'zhijing-attempt-details-全部.csv').exists():
    BASE_DIR = BASE_DIR / 'predataAnalysis'
DATA_FILE = BASE_DIR / 'zhijing-attempt-details-全部.csv'
OPTION_FILE = BASE_DIR / '题库选项.csv'
OUTPUT_DIR = BASE_DIR / '题目筛选与难度输出'
OUTPUT_DIR.mkdir(exist_ok=True)

# 可按研究约定调整；样本不足的题不会被自动建议删除。
MIN_N = 15
FLOOR_MAX = 0.15
CEILING_MIN = 0.85
LONG_TIME_QUANTILE = 0.90
LONG_TIME_RATIO = 1.30
print('数据文件：', DATA_FILE.resolve())
print('输出目录：', OUTPUT_DIR.resolve())

In [ ]:
raw = pd.read_csv(DATA_FILE, encoding='utf-8-sig')
rename = {
    '匿名答卷编号':'attempt_id', '题目编号':'item_id', '题目标题':'title',
    '素材模态':'modality', '能力类型':'ability', '选项形式':'option_type',
    '本题候选情绪':'shown_emotions', '用户选择情绪':'selected_emotions', '标准情绪':'correct_emotions',
    '标签得分率（%）':'label_score', '作答用时（秒）':'response_seconds',
    '作答状态':'status'
}
has_shown_options = '本题候选情绪' in raw.columns
required = set(rename) - {'本题候选情绪'}
missing = sorted(required - set(raw.columns))
assert not missing, f'答卷明细缺少字段：{missing}'
if not has_shown_options:
    print('警告：这是旧版导出文件，缺少“本题候选情绪”。无法准确计算呈现概率或零选择选项；以下选项分析仅基于已出现选项。请从管理员后台重新导出答卷明细。')
    raw['本题候选情绪'] = ''
df = raw.rename(columns=rename).copy()
df['label_score'] = pd.to_numeric(df['label_score'], errors='coerce')
df['response_seconds'] = pd.to_numeric(df['response_seconds'], errors='coerce')
df = df[df['status'].eq('已作答')].copy()
df['is_correct'] = np.isclose(df['label_score'], 100).astype(int)
parse_set = lambda value: {v.strip() for v in str(value).split('｜') if v.strip()} if pd.notna(value) else set()
df['shown_set'] = df['shown_emotions'].apply(parse_set)
df['selected_set'] = df['selected_emotions'].apply(parse_set)
df['correct_set'] = df['correct_emotions'].apply(parse_set)
df['selected_list'] = df['selected_set'].apply(sorted)
df['correct_list'] = df['correct_set'].apply(sorted)
quality_issues = df[has_shown_options & ~df.apply(lambda r: r.correct_set.issubset(r.shown_set), axis=1)]
if len(quality_issues):
    print(f'数据质量警告：{len(quality_issues)} 条作答的标准答案未全部包含在实际候选集合中；分析保留原始数据，不做静默修正。')

# 与原分析保持一致：排除已经标记为重点可疑的答卷；没有处理表时使用全部答卷。
attempt_file = BASE_DIR / '预测试分析输出' / '异常答卷处理表.csv'
if attempt_file.exists():
    attempts = pd.read_csv(attempt_file, encoding='utf-8-sig')
    id_col, include_col = '答卷编号', '纳入主分析'
    if {id_col, include_col}.issubset(attempts.columns):
        included = attempts[include_col].astype(str).str.lower().isin(['true', '1', '是'])
        include_ids = set(attempts.loc[included, id_col].astype(str))
        df = df[df['attempt_id'].astype(str).isin(include_ids)].copy()
print(f'主分析：{df.attempt_id.nunique()} 份答卷，{df.item_id.nunique()} 道题，{len(df)} 条作答记录')

## 1. 天花板题与地板题

难度（通过率）定义为严格答对人数除以该题有效作答人数。默认将通过率不高于 0.15 标记为地板题，不低于 0.85 标记为天花板题。只有有效样本量至少为 `MIN_N` 时才进入自动删除建议；样本不足时只标记为需补测。

In [ ]:
item = (df.groupby('item_id').agg(
    题目标题=('title','first'), 素材模态=('modality','first'),
    能力类型=('ability','first'), 选项形式=('option_type','first'),
    有效作答人数=('attempt_id','nunique'), 答对人数=('is_correct','sum'),
    平均作答时间秒=('response_seconds','mean'), 作答时间中位数秒=('response_seconds','median')
).reset_index().rename(columns={'item_id':'题目编号'}))
item['难度_严格答对率'] = item['答对人数'] / item['有效作答人数']
item['极端效应'] = np.select(
    [item['难度_严格答对率'].le(FLOOR_MAX), item['难度_严格答对率'].ge(CEILING_MIN)],
    ['地板题', '天花板题'], default='无')
item['题目处理建议'] = np.select(
    [item['有效作答人数'].lt(MIN_N), item['极端效应'].ne('无')],
    ['样本不足，补测后判断', '建议删除（须内容复核）'], default='保留进入难度分级')
extreme = item[item['极端效应'].ne('无')].sort_values(['极端效应','难度_严格答对率'])
delete_items = extreme[extreme['有效作答人数'].ge(MIN_N)].copy()
item.to_csv(OUTPUT_DIR/'01_全部题目极端效应.csv', index=False, encoding='utf-8-sig')
delete_items.to_csv(OUTPUT_DIR/'02_建议删除的天花板地板题.csv', index=False, encoding='utf-8-sig')
print('极端题数量：', len(extreme), '；达到样本量要求的建议删除题：', len(delete_items))
display(extreme.round(3))

## 2. 候选选项呈现与选择分析

新版导出按被试实际看到的候选集合计算：条件选择概率 = 看到该选项且选择它的人数 ÷ 看到该选项的人数。不同被试可能看到不同补充候选项，不能统一使用该题总作答人数作分母。多选题各选项的条件选择概率之和可能超过 100%。若输入为缺少 `本题候选情绪` 的旧版 CSV，只保留“仅基于已出现选项”的近似分析，并且不推断呈现次数或零选择选项。

In [ ]:
correct_map = df.groupby('item_id')['correct_set'].first().to_dict()
title_map = df.groupby('item_id')['title'].first().to_dict()
option_rows = []
for item_id, g in df.groupby('item_id'):
    correct = set(correct_map[item_id])
    if has_shown_options:
        options = sorted(set().union(*g['shown_set']))
        for option in options:
            shown = g[g['shown_set'].apply(lambda values: option in values)]
            selected = shown[shown['selected_set'].apply(lambda values: option in values)]
            shown_n, selected_n = shown['attempt_id'].nunique(), selected['attempt_id'].nunique()
            option_rows.append([item_id, title_map[item_id], option, option in correct, '标准答案选项' if option in correct else '非答案候选选项', shown_n, selected_n, selected_n / shown_n if shown_n else np.nan, selected_n == 0])
    else:
        options = sorted(set().union(*g['selected_set'], correct))
        for option in options:
            selected_n = g[g['selected_set'].apply(lambda values: option in values)]['attempt_id'].nunique()
            option_rows.append([item_id, title_map[item_id], option, option in correct, '标准答案选项' if option in correct else '非答案候选选项', np.nan, selected_n, np.nan, False])
option_analysis = pd.DataFrame(option_rows, columns=['题目编号','题目标题','选项','是否标准答案选项','选项性质','呈现人数','选择人数','条件选择概率','是否零选择选项'])
option_analysis['条件选择百分比'] = option_analysis['条件选择概率'] * 100
option_analysis['分析口径'] = '按实际呈现人数' if has_shown_options else '旧版导出：仅基于已出现选项'
option_analysis = option_analysis[['题目编号','题目标题','选项','是否标准答案选项','选项性质','呈现人数','选择人数','条件选择概率','条件选择百分比','是否零选择选项','分析口径']].sort_values(['题目编号','条件选择概率'], ascending=[True,False], na_position='last')
option_analysis.to_csv(OUTPUT_DIR/'03_每题候选选项呈现与选择概率.csv', index=False, encoding='utf-8-sig')

question_rows = []
for item_id, g in df.groupby('item_id'):
    correct = set(correct_map[item_id]); selected_sets = g['selected_set']
    item_options = option_analysis[option_analysis['题目编号'].eq(item_id)]
    answer_probs = item_options.loc[item_options['是否标准答案选项'], '条件选择概率']
    distractor_probs = item_options.loc[~item_options['是否标准答案选项'], '条件选择概率']
    zero = item_options['是否零选择选项'] if has_shown_options else pd.Series(dtype=bool)
    zero_distractors = item_options.loc[~item_options['是否标准答案选项'], '是否零选择选项'] if has_shown_options else pd.Series(dtype=bool)
    question_rows.append({
        '题目编号': item_id, '题目标题': title_map[item_id], '标准答案': '｜'.join(sorted(correct)), '有效作答人数': g['attempt_id'].nunique(),
        '标准答案完整命中率': np.mean([values == correct for values in selected_sets]),
        '标准答案全部被选率': np.mean([correct.issubset(values) for values in selected_sets]),
        '至少一个标准答案被选率': np.mean([bool(values & correct) for values in selected_sets]),
        '标准答案选项平均条件选择概率': answer_probs.mean(),
        '至少一个非答案候选项被选率': np.mean([bool(values - correct) for values in selected_sets]),
        '非答案候选项平均条件选择概率': distractor_probs.mean(),
        '每名被试平均误选的非答案选项数': np.mean([len(values - correct) for values in selected_sets]),
        '零选择候选项数量': int(zero.sum()) if has_shown_options else np.nan,
        '零选择非答案候选项数量': int(zero_distractors.sum()) if has_shown_options else np.nan,
        '分析口径': '按实际呈现人数' if has_shown_options else '旧版导出：仅基于已出现选项'
    })
question_option_summary = pd.DataFrame(question_rows)
for column in ['标准答案完整命中率','标准答案全部被选率','至少一个标准答案被选率','标准答案选项平均条件选择概率','至少一个非答案候选项被选率','非答案候选项平均条件选择概率']:
    question_option_summary[column.replace('率', '百分比')] = question_option_summary[column] * 100
question_option_summary.to_csv(OUTPUT_DIR/'04_每题标准答案与非答案候选项汇总.csv', index=False, encoding='utf-8-sig')
zero_options = option_analysis[option_analysis['是否零选择选项']].copy() if has_shown_options else pd.DataFrame(columns=option_analysis.columns)
print('已输出候选选项层分析', len(option_analysis), '条；题目层汇总', len(question_option_summary), '道。')
if has_shown_options:
    print('实际呈现后零选择候选项数：', len(zero_options))
else:
    print('旧版导出不含候选集合：未计算呈现人数、条件选择概率或零选择选项。')

## 3. 作答时间过长题目

不同素材模态天然耗时不同，因此比较各题的中位作答时间，并在各模态内部判断。默认同时满足以下条件才标记：题目中位时间位于同模态题目的前 10%、至少为同模态典型题中位时间的 1.3 倍、且样本量不少于 `MIN_N`。音频和视频还必须人工核对素材时长；现有导出不含素材时长。

In [ ]:
modality_baseline = (item.groupby('素材模态')['作答时间中位数秒'].agg(
    模态作答中位数秒='median', 模态长时阈值秒=lambda x: x.quantile(LONG_TIME_QUANTILE)).reset_index())
time_result = item.merge(modality_baseline, on='素材模态', how='left')
time_result['相对模态中位数倍数'] = time_result['作答时间中位数秒'] / time_result['模态作答中位数秒']
time_result['作答过久'] = (time_result['有效作答人数'].ge(MIN_N) &
    time_result['作答时间中位数秒'].ge(time_result['模态长时阈值秒']) &
    time_result['相对模态中位数倍数'].ge(LONG_TIME_RATIO))
time_result['时间复核建议'] = np.where(~time_result['作答过久'], '无',
    np.where(time_result['素材模态'].isin(['音频','视频']), '先核对素材时长；若素材不长则精简题干/选项', '检查并精简题干/选项阅读负荷'))
long_items = time_result[time_result['作答过久']].sort_values('相对模态中位数倍数', ascending=False)
time_result.to_csv(OUTPUT_DIR/'05_全部题目作答时间诊断.csv', index=False, encoding='utf-8-sig')
long_items.to_csv(OUTPUT_DIR/'06_作答过久题目复核表.csv', index=False, encoding='utf-8-sig')
print('作答过久题目数：', len(long_items)); display(long_items.round(2))

## 4. 筛题后的难度分级

预测试阶段不计算区分度。先剔除达到样本量要求的天花板/地板题，再按严格答对率分级：`≤0.30` 很难、`(0.30,0.50]` 较难、`(0.50,0.70]` 中等、`(0.70,0.85)` 较易。样本不足的题单独标记为需补测，不强行分级。

In [ ]:
remove_ids = set(delete_items['题目编号'])
difficulty = item[~item['题目编号'].isin(remove_ids)].copy()
difficulty['难度等级'] = np.select(
    [difficulty['有效作答人数'].lt(MIN_N), difficulty['难度_严格答对率'].le(.30),
     difficulty['难度_严格答对率'].le(.50), difficulty['难度_严格答对率'].le(.70)],
    ['样本不足，需补测', '很难', '较难', '中等'], default='较易')
difficulty['区分度'] = '预测试阶段暂不计算'
difficulty.to_csv(OUTPUT_DIR/'07_筛题后题库难度分级.csv', index=False, encoding='utf-8-sig')
summary = {
    '有效答卷数': int(df.attempt_id.nunique()), '分析题目数': int(item.shape[0]),
    '地板题数': int(item['极端效应'].eq('地板题').sum()),
    '天花板题数': int(item['极端效应'].eq('天花板题').sum()),
    '达到样本量要求的建议删除题数': int(len(delete_items)),
    '零选择候选项数_仅新版导出有效': int(len(zero_options)) if has_shown_options else None,
    '作答过久题目数': int(len(long_items)), '筛题后题目数': int(len(difficulty)),
    '参数': {'最小样本量':MIN_N, '地板上限':FLOOR_MAX, '天花板下限':CEILING_MIN,
             '长时百分位':LONG_TIME_QUANTILE, '长时相对倍数':LONG_TIME_RATIO}
}
(OUTPUT_DIR/'分析摘要.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.Series(summary).to_frame('结果'))
display(difficulty['难度等级'].value_counts().to_frame('题目数'))
print('已生成：'); [print(' -', p.name) for p in sorted(OUTPUT_DIR.iterdir())]